# Journey of a Request

A user tries to create an account
```http
POST /users
```

With the object:
```json
{
     "username"         : "Daniel"
    ,"password"         : "Password123!"
    ,"confirm_password" : "Password123!"
}

## 1. Basemodel
What format do we expect/accept the data to be?

Using `pydantic` we can set this:
```py
from pydantic import Basemodel

class UserCreate(Basemodel):
    username    : str
    password    : str
```

Now FastAPI expects:
* username as a string 
* password as a string

This gives us:
* validation
* documentation
* type safety
* OpenAPI generation

`BaseModel` acts a contract:
```
 Client
   │
   ▼
BaseModel
   │
   ▼
 Route
```

## 2. model_validator
`BaseModel` validates individual fields.<br>
```json
{
    "password"              : "789"
    ,"confirm_password"     : "123"
}
```

Whilst both fields are valid strings, together they are invalid.
`model_validator` allows use to validate dependent fields.
```py
from pydantic import BaseModel, model_validator

class RegisteredUser(BaseModel):
    password            : str
    confirm_password    : str

    @model_validator(mode="after")
    def validate_passwords(self):
        if self.password != self.confirm_password:
            raise ValueError(
                "Passwords do not match"
            )
        return self
```

`model_validator` bundles the data together to validate it:
```
BaseModel
   │
   ├─ Is username a string?
   ├─ Is age an integer?
   │
   ▼
model_validator
   │
   └─ Do all the fields make sense together?
```

## 3. HTTPException
Validation succeeded and routing starts.<br>
A user requests a user does not exist:
```http
GET /users/999
```

We use:
```py
raise HTTPException(
    status_code = 404
    ,detail     = "User not found"
)
```
Even though the data is _valid_, it is a business problem.

```
       Bad Data Structure?
              ↓
        Validation Error

Valid Data But Business Problem?
              ↓
        HTTPException
``

## 4. RequestValidationError Handler
When validation fails, FastAPI automatically returns:
```json
{
    "detail"    : [...]
}
```

Business requires more detail:
```json
{
     "success"  : false
    ,"error"    : "Validation failed"
}
```
For ALL validation errors.

We can create a global validation handler:
```py
@app.execption_handler(
    RequestValidationError
)
```

We are able to log details of errors at a granular level.
```
Pydantic Validation Fails
          │
          ▼
  RequestValidationError
          │
          ▼
  Custom Response Format
```

## 5. Exception Handlers
With an excpected issues (e.g. crash or connection lost), we can use
```py
@app.excepetion_handler(Exception)
```

Dealing with the expected and unexpected issues:
```
Expected Problem    →   HTTPException

Unexpected Problem  →   Exception Handler
```

## 6. Request-ID Middleware
If we recieve a report where _something failed_.<br>
There is zero detail with included as part of the failure.

By generating an ID for every request, we are able to track any issues that arise.<br>
It is used for:
* debugging
* distributed systems
* tracing

```
 Request Arrives
        │
        ▼
Request-ID Middleware
        │
        ▼
 Attach Unique ID
```

## 7. Timing Middleware
If the application is slow, we can check meaure execution times.
```
Request starts  →   Route executes  →   Request ends

X-Process-Time: 0.045 
```

```
Request-ID answers: "Which request?"

Timing answers:     "How long?"
```

## 8. CORSMiddleware
Frontend is accessing Port 3000 / Backend is using Port 8000

Browser returns: 
> Blocked by CORS policy

By default, the browser protects users where the origins are different.

`CORSMiddleware` allows trusted origins
```
Frontend    →   Browser Checks CORS →   API
```

> CORS is not API security<br>
> CORS is **browser** security

## 9. Security Headers
Browsers can be tricked into unsafe behaviour

**HTTP Strict-Transport-Security (HSTS)**
```
Header:                             Meaning:
Strict-Transport-Security           Only use HTTPS
```
Browsers can treat `text/plain` as HTML or JavaScript, enabling stored XSS or other injection attacks.

**X-Content-Type-Options**
```
Header                              Meaning:
X-Content-Type-Options: nosniff     Do not guess a file type.
                                    Use the declared type.
```
Browsers strictly obey the delared `Content-type`, therefore `text/plain` is treated as is, not as `HTML` or `JavaScript`
 
This is usually implemented in middleware.
```
Security headers    →   Extra instructions sent to browsers

## 10. Upload file
Users want to upload:
```
profile_pic.png
resume.pdf
invoice.xlsx
```

Instead of using `file: bytes`,<br>
Use `file: UploadFile`

This allows streaming of the data, less memory, and can include metadata.

## 11. Local Storage
The uploaded files need to go somewhere.

One solution is to local server/drive storage.

## 12. S3 Storage
Another solution is AWS' Simple Storage Service:
- designed for massive scalability, 
- 11‑nines durability, 
- 99.99% availability, 
- storing any amount of data as objects inside buckets.

## 13. Cross-Site Request Forgery (CSRF) Double Submit
If an attacker creates
```
<form action="https://bank.com/transfer>
```
The browser automicaticallyt sends cookies and money gets transferred.

This is Cross-Site Request Forgery.

**Double Submit Solution**
Server sends the cookie: `csrf_token=abc=123`

Frontend sends the header: `X-CRSF-Token: abc123`

Server checks the Cookie Token and the Header Token.<br>
If they match, the request is allowed.

This is ususally needed when **authentication uses cookies**<br>
But not required when using JWT tokens (stored outside cookies).

## 14. slowapi Rate Limiting
If an attacker sends 10,000 requests to a login endpoint we want to able to restrict this as much as possible.

We can implement a maximum number of requests of 5 per minute.<br>
On the 6th request, we can return `429 Too Many Requests`